In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb
import implicit
import scipy.sparse as sparse
import mlflow
import mlflow.xgboost
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load merged data
data = pd.read_csv("../data/amazon_beauty_merged.csv")
print(data.shape)
print(data.columns.tolist())

(701528, 14)
['rating', 'title_x', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'title_y', 'description', 'price', 'average_rating']


## Data Cleaning


In [4]:
#only user with 2 review and above 
user_counts = data.groupby("user_id").size()
valid_users = user_counts[user_counts >= 2].index
data = data[data["user_id"].isin(valid_users)]

#only products with at least 3 reviews
product_counts = data.groupby("parent_asin").size()
valid_products = product_counts[product_counts >= 3].index
data = data[data["parent_asin"].isin(valid_products)]

print("After filtering:")
print("Shape:", data.shape)
print("Unique users:", data["user_id"].nunique())
print("Unique products:", data["parent_asin"].nunique())

After filtering:
Shape: (75301, 14)
Unique users: 39831
Unique products: 10532


## Ecoding users and products

In [5]:
user_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data["user_idx"] = user_encoder.fit_transform(data["user_id"])
data["product_idx"] = product_encoder.fit_transform(data["parent_asin"])

n_users = data["user_idx"].nunique()
n_products = data["product_idx"].nunique()

print(f"Users: {n_users}")
print(f"Products: {n_products}")

Users: 39831
Products: 10532


## Performing feature Engineering

In [6]:
# for User features
user_features = data.groupby("user_idx").agg(
    user_avg_rating=("rating", "mean"),
    user_review_count=("rating", "count"),
    user_rating_std=("rating", "std")
).fillna(0).reset_index()

# for Product features
product_features = data.groupby("product_idx").agg(
    product_avg_rating=("rating", "mean"),
    product_review_count=("rating", "count"),
    product_rating_std=("rating", "std")
).fillna(0).reset_index()

# lets Merge features back
data = data.merge(user_features, on="user_idx", how="left")
data = data.merge(product_features, on="product_idx", how="left")

# for Interaction features
data["rating_deviation"] = data["rating"] - data["product_avg_rating"]
data["review_length"] = data["text"].fillna("").apply(len)
data["is_verified"] = data["verified_purchase"].astype(int)

print("Features created successfully")
print(data[["user_avg_rating", "product_avg_rating", "rating_deviation", 
            "review_length", "is_verified"]].describe())

Features created successfully
       user_avg_rating  product_avg_rating  rating_deviation  review_length  \
count     75301.000000        75301.000000      7.530100e+04   75301.000000   
mean          4.122229            4.122229      2.264648e-18     249.261829   
std           1.155042            0.720206      1.130755e+00     369.944896   
min           1.000000            1.000000     -3.800000e+00       0.000000   
25%           3.500000            3.770833     -4.666667e-01      54.000000   
50%           4.545455            4.272727      2.307692e-01     134.000000   
75%           5.000000            4.647059      6.923077e-01     298.000000   
max           5.000000            5.000000      3.428571e+00   10205.000000   

        is_verified  
count  75301.000000  
mean       0.757453  
std        0.428626  
min        0.000000  
25%        1.000000  
50%        1.000000  
75%        1.000000  
max        1.000000  


## Train XGBoost Model

In [14]:
feature_cols = [
    "user_avg_rating", "user_review_count", "user_rating_std",
    "product_avg_rating", "product_review_count", "product_rating_std",
     "review_length", "is_verified" ]

X = data[feature_cols]
y = data["rating"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

with mlflow.start_run(run_name="xgboost_baseline"):
    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)
    
    mlflow.log_metric("rmse", rmse)
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.xgboost.log_model(model, "xgboost_model")
    
    print(f"RMSE: {rmse:.4f}")
    print("Model logged to MLflow")

Train: (60240, 8), Test: (15061, 8)


2026/05/26 17:13:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE: 0.5647
Model logged to MLflow


In [15]:
import mlflow
print(mlflow.get_tracking_uri())

sqlite:///C:/Users/Ronak/OneDrive/Desktop/GENAI/GenAI-Powered%20Product%20Recommendation%20System%20Using%20Amazon%20Reviews/notebooks/mlflow.db


## Performing Implicit ALS

In [16]:
item_user = sparse.csr_matrix(
    (data["rating"].astype(float),
     (data["product_idx"], data["user_idx"]))
)

print("Matrix shape:", item_user.shape)
print("Sparsity: {:.4f}%".format(100 * (1 - item_user.nnz / (item_user.shape[0] * item_user.shape[1]))))

# Train ALS model
als_model = implicit.als.AlternatingLeastSquares(
    factors=50,
    iterations=20,
    regularization=0.1,
    random_state=42
)
als_model.fit(item_user)
print("ALS model trained successfully")

Matrix shape: (10532, 39831)
Sparsity: 99.9835%


  0%|          | 0/20 [00:00<?, ?it/s]

ALS model trained successfully


## Saving Model

In [17]:
import pickle
import os

os.makedirs("../ml", exist_ok=True)

# Save XGBoost
model.save_model("../ml/xgboost_model.json")

# Save ALS
with open("../ml/als_model.pkl", "wb") as f:
    pickle.dump(als_model, f)

# Save encoders
with open("../ml/user_encoder.pkl", "wb") as f:
    pickle.dump(user_encoder, f)

with open("../ml/product_encoder.pkl", "wb") as f:
    pickle.dump(product_encoder, f)

# Save filtered data
data.to_csv("../data/amazon_beauty_processed.csv", index=False)

print("All models and encoders saved successfully")

All models and encoders saved successfully


## Comparing model and loggin MLflow

In [21]:
df_meta = pd.read_json("../data/meta_All_Beauty.jsonl", lines=True)

# Create product lookup from metadata directly
product_lookup = df_meta.set_index("parent_asin")["title"].to_dict()

# ALS quick evaluation
user_item = item_user.T.tocsr()

user_id = 0
recommended = als_model.recommend(user_id, user_item[user_id], N=5)
print("ALS top 5 recommendations for user 0:")

valid_product_indices = set(range(len(product_encoder.classes_)))

for product_idx, score in zip(recommended[0], recommended[1]):
    if product_idx not in valid_product_indices:
        continue
    product_asin = product_encoder.inverse_transform([product_idx])[0]
    title = product_lookup.get(product_asin, "Unknown Product")
    print(f"  Score: {score:.3f} | {title[:60]}")

# Summary
print("\n=== MODEL COMPARISON ===")
print(f"XGBoost RMSE:      {rmse:.4f} (feature-based ranking)")
print(f"ALS factors:       50 (collaborative filtering)")
print(f"Training users:    {n_users}")
print(f"Training products: {n_products}")
print(f"Matrix sparsity:   ~99%")
print("\nConclusion: XGBoost handles cold-start better due to features.")
print("ALS handles user-item patterns better for active users.")
print("Both used together = hybrid system (your project strength)")

ALS top 5 recommendations for user 0:
  Score: 0.010 | Original Detangler Hair Brush

=== MODEL COMPARISON ===
XGBoost RMSE:      0.5647 (feature-based ranking)
ALS factors:       50 (collaborative filtering)
Training users:    39831
Training products: 10532
Matrix sparsity:   ~99%

Conclusion: XGBoost handles cold-start better due to features.
ALS handles user-item patterns better for active users.
Both used together = hybrid system (your project strength)
